In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
sp.init_printing(use_latex='mathjax')  # LaTeX in Jupyter
from IPython.display import display

repl/_repl_lut_sincos.py
LUT sin/cos in Python/numpy. Same logic as lut_sincos.cpp.

In [ ]:
import numpy as np
import time

LUT_N = 1024
theta_lut = np.linspace(0, 2*np.pi, LUT_N + 1, dtype=np.float32)
lut_sin = np.sin(theta_lut).astype(np.float32)
lut_cos = np.cos(theta_lut).astype(np.float32)

def lut_lookup(theta, table):
    """Linear interpolation LUT lookup. theta: any range float array."""
    idx_f = (theta % (2*np.pi)) * (LUT_N / (2*np.pi))
    idx_i = idx_f.astype(np.int32) % LUT_N
    frac  = idx_f - np.floor(idx_f)
    idx_n = (idx_i + 1) % LUT_N
    return table[idx_i] + frac * (table[idx_n] - table[idx_i])

def dispersion_lut(N, D):
    nu    = np.fft.fftfreq(N).astype(np.float32)
    phase = (np.pi * D * nu**2).astype(np.float32)
    return lut_lookup(phase, lut_cos) + 1j * lut_lookup(phase, lut_sin)

def dispersion_std(N, D):
    nu    = np.fft.fftfreq(N).astype(np.float32)
    phase = np.pi * D * nu**2
    return np.exp(1j * phase)

print("=" * 55)
print("LUT SIN/COS  (N=%d entries, %.1f KB)" % (LUT_N, 2*LUT_N*4/1024))
print("=" * 55)
print()

# accuracy
N = 512; D = 5000
H_lut = dispersion_lut(N, D)
H_std = dispersion_std(N, D)
err = np.max(np.abs(H_lut - H_std))
theory = np.pi**2 / (2 * LUT_N**2)
print(f"Dispersion kernel N={N} D={D}:")
print(f"  max |H_lut - H_std| = {err:.2e}")
print(f"  theory O(pi^2/2N^2) = {theory:.2e}")
print()

# benchmark
reps = 50000
t0 = time.perf_counter()
for _ in range(reps): dispersion_lut(N, D)
ms_lut = (time.perf_counter()-t0)/reps*1000

t0 = time.perf_counter()
for _ in range(reps): dispersion_std(N, D)
ms_std = (time.perf_counter()-t0)/reps*1000

print(f"Benchmark ({reps} reps, N={N}):")
print(f"  np.exp(1j*phase): {ms_std:.4f} ms/call")
print(f"  LUT interp:       {ms_lut:.4f} ms/call")
print(f"  Ratio:            {ms_lut/ms_std:.2f}x  "
      f"({'LUT faster' if ms_lut < ms_std else 'exp faster -- vectorized exp wins in numpy'})")
print()
print("Note: numpy exp() is already vectorized SIMD -> hard to beat with LUT in Python.")
print("LUT wins in C/FPGA/MCU where exp() is software. In Python: use numpy directly.")